In [1]:
import json
import math
import os.path
from pathlib import Path
import torch
import numpy as np
import pandas as pd

In [2]:
np_rng = np.random.default_rng(1894327)
train_fraction = 0.8

In [3]:
source_path = Path("D:\\TruthIsUniversal_In_Phi_3_5_Mini\\activations_for_full_sequences")

dest_path = Path("D:\\TruthIsUniversal_In_Phi_3_5_Mini\\activations_for_final_token_of_sequences")
dest_path.mkdir(exist_ok=True)

In [4]:
dsets_folder = Path("true_false_datasets")

In [5]:
dsets_index_df = pd.read_csv(dsets_folder / "categ_dataset_pairs.csv", index_col="Idx")

In [6]:
# dsets_index_df

In [7]:
#dict of dset idx (in dsets_index_df) to list of record indices 
train_split_record_idxs: dict[int, list[int]]= {}
validation_split_record_idxs: dict[int, list[int]]= {}

#dict of dset idx (in dsets_index_df) to list of length-in-tokens for each record of that dataset
record_lengths_in_tokens: dict[int, list[int]] = {}

In [8]:
def slice_diff_seq_pos_for_each_record(hidden_states_for_all_seq_pos, seq_pos_for_each_record):
    assert len(seq_pos_for_each_record) == hidden_states_for_all_seq_pos.shape[0]
    num_records = hidden_states_for_all_seq_pos.shape[0]
    hidden_state_size = hidden_states_for_all_seq_pos.shape[2]
    states_for_last_tok_in_records = torch.zeros((num_records, hidden_state_size))
    for record_idx in range(len(seq_pos_for_each_record)):
        states_for_last_tok_in_records[record_idx, :] = hidden_states_for_all_seq_pos[record_idx, seq_pos_for_each_record[record_idx], :]
    
    return states_for_last_tok_in_records

In [9]:
for dset_idx, row in dsets_index_df.iterrows():
    categ_nm = row["Categ_Folder"]
    dset_file_nm = row["Dataset_File"]
    dset_nm = os.path.splitext(dset_file_nm)[0]
    harvest_result_path = source_path / categ_nm / (dset_nm + ".pt")
    
    categ_output_folder = dest_path / categ_nm
    categ_output_folder.mkdir(exist_ok=True)
    
    activations_harvest_result = torch.load(harvest_result_path, weights_only=True)
    
    token_lengths = activations_harvest_result['token_lengths'].tolist()
    dset_size = len(token_lengths)
    
    idxs_of_last_tok_in_seqs = activations_harvest_result['token_lengths'] - 1
    
    post6_states = activations_harvest_result['post6_hidden_states']
    post6_last_token_states = slice_diff_seq_pos_for_each_record(post6_states, idxs_of_last_tok_in_seqs)
    activations_harvest_result['post6_hidden_states'] = None
    del post6_states
    torch.save(post6_last_token_states, categ_output_folder / (dset_nm + "_post6.pt"))
    del post6_last_token_states
    
    post16_states = activations_harvest_result['post16_hidden_states']
    post16_last_token_states = slice_diff_seq_pos_for_each_record(post16_states, idxs_of_last_tok_in_seqs)
    activations_harvest_result['post16_hidden_states'] = None
    del post16_states
    torch.save(post16_last_token_states, categ_output_folder / (dset_nm + "_post16.pt"))
    del post16_last_token_states
    
    post22_states = activations_harvest_result['post22_hidden_states']
    post22_last_token_states = slice_diff_seq_pos_for_each_record(post22_states, idxs_of_last_tok_in_seqs)
    activations_harvest_result['post22_hidden_states'] = None
    del post22_states
    torch.save(post22_last_token_states, categ_output_folder / (dset_nm + "_post22.pt"))
    del post22_last_token_states    
    
    post29_states = activations_harvest_result['post29_hidden_states']
    post29_last_token_states = slice_diff_seq_pos_for_each_record(post29_states, idxs_of_last_tok_in_seqs)
    activations_harvest_result['post29_hidden_states'] = None
    del post29_states
    torch.save(post29_last_token_states, categ_output_folder / (dset_nm + "_post29.pt"))
    del post29_last_token_states
    
    
    record_lengths_in_tokens[dset_idx] = token_lengths
    
    
    train_size = math.floor(dset_size*train_fraction)
    train_split_indices = sorted(np_rng.choice(dset_size, train_size, replace=False).tolist())
    validation_split_indices = sorted(list(set(range(dset_size)) - set(train_split_indices)))
    train_split_record_idxs[dset_idx] = train_split_indices
    validation_split_record_idxs[dset_idx] = validation_split_indices

In [10]:
train_split_records_path = dsets_folder / "dset_record_token_lengths.json"
with train_split_records_path.open("w") as f:
    json.dump(record_lengths_in_tokens, f)
train_split_records_path = dsets_folder / "train_split_record_indices.json"
with train_split_records_path.open("w") as f:
    json.dump(train_split_record_idxs, f)
validation_split_records_path = dsets_folder / "validation_split_record_indices.json"
with validation_split_records_path.open("w") as f:
    json.dump(validation_split_record_idxs, f)